# Strong Allee effect and harvesting

Assembled from five task fragments. Read the scope/status in each section. Synthetic examples are not empirical observations. The final submission requires completed derivations, algorithms, results, interpretation and AI/data acknowledgements.

In [ ]:
import os
from pathlib import Path

import mm_allee

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists() and (path / "src/mm_allee").exists()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the project checkout")
os.chdir(ROOT)
print("Package:", mm_allee.__version__)

## 1. Foundations and Model A: steady populations under a fixed catch

<div style="border-left:5px solid #2a78d6;background:#2a78d614;padding:10px 14px;border-radius:4px"><strong>Question answered in this section</strong><br>If we keep catching the same amount every year, at which population levels can the stock stay steady, and does it return to those levels after a small disturbance? Where is the largest catch that still allows a steady population?</div>

**Owner:** WP01. **Data type:** analytical derivations and numerical root finding/simulation of the stated ODEs; no field observations are used.
The same functions generate the report figures through `python experiments/wp01/run.py --config configs/wp01.json --output results/wp01`.

### 1.1 Model and assumptions

We model one closed, well-mixed population $N(t)\ge 0$ in continuous time with constant parameters $r>0$, $K>0$, $0<A<K$:

$$\text{Logistic: } \frac{dN}{dt}=rN\Big(1-\frac{N}{K}\Big)-H,\qquad
\text{Model A: } \frac{dN}{dt}=rN\Big(1-\frac{N}{K}\Big)\Big(\frac{N}{A}-1\Big)-H,\qquad
\text{Model B: } \frac{dN}{dt}=rN\Big(1-\frac{N}{K}\Big)\Big(\frac{N}{A}-1\Big)-hN.$$

- The factor $(N/A-1)$ is the **strong Allee effect**: growth is negative for $0<N<A$, so $A$ is a minimum viable population.
- $H$ (population/time) is a fixed quota; $h$ (1/time) is a fixed harvesting effort.
- Deterministic, unstructured, no migration; parameters constant.
- **Boundary convention:** a quota cannot be taken from an extinct stock, so when $N$ reaches $0$ the simulation stops (absorbing extinction).

<div style="border-left:5px solid #eb6834;background:#eb683414;padding:10px 14px;border-radius:4px"><strong>Why it matters</strong><br>A threshold population plus a fixed catch can make a stock collapse suddenly and fail to recover, the pattern seen in several marine fisheries.</div>

### 1.2 Nondimensionalisation

Set $x=N/K$, $\tau=rt$, $a=A/K\in(0,1)$, $q=H/(rK)$, $e=h/r$. The chain rule gives
$\dfrac{dN}{dt}=\dfrac{d(Kx)}{d\tau}\dfrac{d\tau}{dt}=rK\dfrac{dx}{d\tau}$, so dividing by $rK$:

$$\frac{dx}{d\tau}=G(x)-q,\qquad G(x)=x(1-x)\Big(\frac{x}{a}-1\Big).$$

Four dimensional parameters collapse to two shape parameters, $a$ and $q$. The cell below checks numerically that the dimensional right-hand side equals $rK$ times the dimensionless one.

In [ ]:
import numpy as np

from mm_allee.models import Parameters, dimensional_rhs, rhs

p = Parameters(
    r=0.5, K=1000.0, A=200.0, H=150.0
)  # illustrative values: 1/yr, tonnes, tonnes, tonnes/yr
d = p.dimensionless()
N = np.array([0.0, 100.0, 200.0, 500.0, 800.0, 1000.0])
lhs = dimensional_rhs(N, "A", p)
scaled = p.r * p.K * rhs(N / p.K, "A", a=d["a"], harvest=d["q"])
print("a = A/K =", d["a"], "  q = H/(rK) =", d["q"])
print("dN/dt (t/yr):       ", np.round(lhs, 6))
print("rK * dx/dtau (t/yr):", np.round(scaled, 6))
print("max difference:", np.max(np.abs(lhs - scaled)))

### 1.3 Equilibria and stability: the method

<div style="border-left:5px solid #1baf7a;background:#1baf7a14;padding:10px 14px;border-radius:4px"><strong>Definitions</strong><br>An <b>equilibrium</b> $x_e$ solves $f(x_e)=0$: the population stays put. <b>Stability</b> asks what a small disturbance does. In one dimension the sign of $f$ on each side decides: $f>0$ below and $f<0$ above means <i>stable</i>; the reverse means <i>unstable</i>; the same sign on both sides means <i>semistable</i> (attracts from one side only).</div>

When $f'(x_e)\neq0$, linearising $\xi=x-x_e$ gives $\dot\xi\approx f'(x_e)\xi$: stable if $f'<0$, unstable if $f'>0$, with relaxation time $1/|f'(x_e)|$.
When $f'(x_e)=0$ the linearisation says nothing, and only the one-sided signs decide.

### 1.4 Logistic baseline with constant yield

$f(x)=x(1-x)-q=0 \Rightarrow x_\pm=\tfrac12\big(1\pm\sqrt{1-4q}\big)$ and $f'(x_\pm)=1-2x_\pm=\mp\sqrt{1-4q}$.

| harvest | equilibria | stability |
|---|---|---|
| $q=0$ | $0,\ 1$ | $0$ unstable ($f'=1$), $1$ stable |
| $0<q<\tfrac14$ | $x_-<x_+$ | $x_-$ unstable (harvest-made threshold), $x_+$ stable |
| $q=\tfrac14$ | $x=\tfrac12$ (double) | semistable: $f=-(x-\tfrac12)^2\le0$ |
| $q>\tfrac14$ | none | every population collapses |

So the maximum constant yield is $H_c=rK/4$ at $N=K/2$.

In [ ]:
from mm_allee.equilibria import equilibria_logistic

for q in [0.0, 0.1, 0.25, 0.3]:
    recs = equilibria_logistic(q)
    text = (
        ", ".join(f"x={r.x:.4f} ({r.stability}, f'={r.derivative:+.3f})" for r in recs)
        or "none: collapse"
    )
    print(f"q={q:<5} -> {text}")

### 1.5 Model A: shape of the production curve and the fold

$$G(x)=-\frac{x^3}{a}+\frac{(1+a)x^2}{a}-x,\qquad G'(x)=-\frac{3x^2}{a}+\frac{2(1+a)x}{a}-1.$$

- Signs of the factors: $G<0$ on $(0,a)$, $G>0$ on $(a,1)$, $G<0$ for $x>1$, $G>0$ for $x<0$.
- $G'(x)=0 \iff 3x^2-2(1+a)x+a=0$. Since $G'(a)=1-a>0$ and $G'(1)=(a-1)/a<0$, the larger root lies in $(a,1)$ and is the maximum:

$$x^*=\frac{1+a+\sqrt{1-a+a^2}}{3},\qquad q_c=G(x^*)=\frac{2(1-a+a^2)^{3/2}+(1+a)(2a-1)(a-2)}{27a}.$$

Dimensionally $N^*=Kx^*$ and $H_c=rKq_c$. The cell checks the closed form against an **independent** numerical maximisation.

In [ ]:
from scipy.optimize import minimize_scalar

from mm_allee.equilibria import fold_a
from mm_allee.models import growth

for a in [0.1, 0.2, 0.3, 0.5]:
    x_star, q_c = fold_a(a)
    best = minimize_scalar(
        lambda x: -growth(x, a), bounds=(a, 1), method="bounded", options={"xatol": 1e-12}
    )
    print(
        f"a={a}: formula x*={x_star:.6f}, q_c={q_c:.8f} | numerical max x={best.x:.6f}, G={-best.fun:.8f}"
    )

### 1.6 Model A: the four harvest regimes

Equilibria solve $G(x)=q$. Because $G$ rises on $(a,x^*)$ and falls on $(x^*,1)$:

| regime | physical equilibria | stability |
|---|---|---|
| $q=0$ | $0,\ a,\ 1$ | stable, **unstable (minimum viable population)**, stable |
| $0<q<q_c$ | $x_-\in(a,x^*)$, $x_+\in(x^*,1)$ | $x_-$ unstable, $x_+$ stable; the third cubic root is **negative** and excluded |
| $q=q_c$ | $x^*$ (double root) | **semistable**: $f<0$ on both sides, attracts from above only |
| $q>q_c$ | none | collapse from every start |

<div style="border-left:5px solid #eb6834;background:#eb683414;padding:10px 14px;border-radius:4px"><strong>Zero is not an equilibrium when q > 0</strong><br>$f(0)=-q<0$. Below $x_-$ the population keeps falling with speed at least $q$ near zero and hits $x=0$ in finite time. We stop there (absorbing extinction); continuing the ODE would give negative populations.</div>

### 1.7 Algorithm for the roots

<div style="border-left:5px solid #6250d6;background:#6250d614;padding:10px 14px;border-radius:4px"><strong>Algorithm (implemented in mm_allee.equilibria)</strong><br>1. Compute $x^*, q_c$ from the closed form.<br>2. If $|q-q_c|\le 10^{-12}$: report one double root $x^*$.<br>3. If $0<q<q_c$: $f$ changes sign on $[a,x^*]$ and on $[x^*,1]$; refine each root with Brent's method (bracketing + bisection + inverse quadratic interpolation), bracket width $10^{-14}$.<br>4. Optionally find the negative root on $[-2^k,0]$ and label it <i>excluded</i>.<br>5. Classify each root from the signs of $f(x_e\pm\delta)$, $\delta=\min(10^{-4},\text{gap}/4)$; report $f'(x_e)$ and the residual $|f(x_e)|$.</div>

Why not a generic polynomial solver? Near the fold it returns a complex pair with a tiny imaginary part, as the cell below shows.

In [ ]:
from mm_allee.equilibria import equilibria_a

a = 0.2
x_star, q_c = fold_a(a)
print("numpy.roots at q = q_c:", np.roots([-1 / a, (1 + a) / a, -1.0, -q_c]))
for q in [0.0, 0.3, q_c - 1e-10, q_c, 0.63]:
    print(f"\nq = {q:.12f}")
    for r in equilibria_a(a, q, include_excluded=True):
        print(
            f"  x={r.x:+.8f}  {r.stability:<10} mult={r.multiplicity}  f'={r.derivative:+.3e}  "
            f"signs=({r.left_sign},{r.right_sign})  residual={r.residual:.1e}"
        )

### 1.8 Geometry, phase lines and branch diagram

Equilibria are where the production curve meets the horizontal harvest line. Filled circles are stable, open circles unstable, half-filled semistable.

In [ ]:
import matplotlib.pyplot as plt

from mm_allee.equilibria_plots import branch_figure, perturbation_figure, production_figure

fig = production_figure(a=0.2, logistic_q=(0.1, 0.25, 0.35), allee_q=(0.25 * q_c, q_c, 1.2 * q_c))
plt.show()

**Interpretation.** In the logistic panel the population only has a threshold when harvested. In Model A the threshold exists even at zero harvest ($x=a$), and the harvest line moves it up. At $q=q_c$ the line touches the top of the hill: one semistable point. Above it nothing can be sustained.

In [ ]:
fig = branch_figure(a=0.2)
plt.show()

**Interpretation.** The stable branch (solid) and the collapse threshold (dashed) approach each other as the quota rises and meet at the fold $(q_c,x^*)$, a saddle-node bifurcation. Implicit differentiation of $G(x)=q$ gives $dx/dq=1/G'(x)$: negative on the upper branch, positive on the lower one. The safety margin $x_+-x_-$ shrinks like $\sqrt{q_c-q}$.

### 1.9 Checking the stability labels by simulation

Independent check: start $\delta=0.01$ above and below each equilibrium and integrate with the shared RK45 solver (event at $x=0$).

In [ ]:
fig = perturbation_figure(a=0.2, q=0.3, delta=0.01, tau_end=12.0)
plt.show()

**Interpretation.** Both starts near $x_+$ return to it (stable). Just above $x_-$ the stock recovers to $x_+$; just below, it falls and reaches zero in finite time (unstable threshold). This is the behaviour the sign test predicted.

<div style="border-left:5px solid #1baf7a;background:#1baf7a14;padding:10px 14px;border-radius:4px"><strong>Main result</strong><br>For a fixed quota, the persistent state and the collapse threshold move together and merge at the maximum quota $q_c$. Harvesting near $q_c$ leaves almost no safety margin and very slow recovery, so a small shock can cause collapse.</div>

<div style="border-left:5px solid #eb6834;background:#eb683414;padding:10px 14px;border-radius:4px"><strong>Limitations</strong><br>Deterministic and unstructured (no demographic noise, which matters most near the threshold); phenomenological cubic growth; constant parameters; the quota is assumed to be taken exactly whenever $N>0$. Dimensional numbers are illustrative only.</div>

## 2. Model B, comparison and harvesting history

<div style="border-left:4px solid #2563eb;padding:10px"><strong>SCAFFOLD_ONLY — implementation pending for WP02.</strong> B roots/stability; fair A/B comparisons; up/down ramp with state carry-over; explicit restocking; no invented automatic recovery.</div>

Replace with the completed mathematics, equations, algorithm and explanation **before** code. Label synthetic observations, axes and units; interpret every result. No final experiment is implemented in this placeholder.

In [ ]:
print("WP02: task not implemented yet; see docs/tasks/")

## 3. Collapse thresholds, extinction and basins

<div style="border-left:4px solid #2563eb;padding:10px"><strong>SCAFFOLD_ONLY — implementation pending for WP03.</strong> Threshold derivations; finite versus asymptotic extinction; quadrature/event validation; basin maps; synthetic reference provenance.</div>

Replace with the completed mathematics, equations, algorithm and explanation **before** code. Label synthetic observations, axes and units; interpret every result. No final experiment is implemented in this placeholder.

In [ ]:
print("WP03: task not implemented yet; see docs/tasks/")

## 4. Forward PINNs

<div style="border-left:4px solid #2563eb;padding:10px"><strong>SCAFFOLD_ONLY — implementation pending for WP04.</strong> Differentiable network and residual; initial condition/physics loss; optimiser/collocation algorithm; both models and independent solver errors/runtime.</div>

Replace with the completed mathematics, equations, algorithm and explanation **before** code. Label synthetic observations, axes and units; interpret every result. No final experiment is implemented in this placeholder.

In [ ]:
print("WP04: task not implemented yet; see docs/tasks/")

## 5. Inverse PINNs and validation

<div style="border-left:4px solid #2563eb;padding:10px"><strong>SCAFFOLD_ONLY — implementation pending for WP05.</strong> Identifiable staged unknowns; noisy observations; ODE least-squares baseline; held-out prediction; near-collapse failures; AI/data acknowledgement.</div>

Replace with the completed mathematics, equations, algorithm and explanation **before** code. Label synthetic observations, axes and units; interpret every result. No final experiment is implemented in this placeholder.

In [ ]:
print("WP05: task not implemented yet; see docs/tasks/")